# 03 · Indicadores y síntesis ejecutiva (Estación 2)

**Objetivo.** Calcular los indicadores por categoría y flujo, ordenar los socios por valor y por cantidad para mostrar que **valor ≠ cantidad**, y cerrar con la síntesis ejecutiva para la Gerencia General de Manaco (3 hallazgos, 2 limitaciones, 1 pregunta estratégica).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

sys.path.append(str(Path.cwd().parent / "src"))
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 200
pd.options.display.max_columns = 30

from carga import cargar

df, _ = cargar()

In [2]:
ind = df.groupby(["Código_SA6", "Flujo_comercial"]).agg(
    valor_total_musd=("Valor_USD", lambda s: s.sum() / 1e6),
    pares_totales_millones=("Cantidad_pares", lambda s: s.sum() / 1e6),
    socios_distintos=("Socio_comercial", "nunique"),
    registros=("Valor_USD", "size"),
)
agr = df.groupby(["Código_SA6", "Flujo_comercial"])
ind["valor_par_ponderado"] = agr["Valor_USD"].sum() / agr["Cantidad_pares"].sum()
ind["valor_par_promedio_simple"] = agr["Valor_USD_por_par"].mean()
ind["participacion_valor_pct"] = ind["valor_total_musd"] / ind.groupby("Flujo_comercial")["valor_total_musd"].transform("sum") * 100
ind["participacion_pares_pct"] = ind["pares_totales_millones"] / ind.groupby("Flujo_comercial")["pares_totales_millones"].transform("sum") * 100
print("Indicadores por categoría SA6 y flujo (participación % calculada dentro de cada flujo):")
display(ind.round(2))

Indicadores por categoría SA6 y flujo (participación % calculada dentro de cada flujo):


valor_total_musd  pares_totales_millones  socios_distintos  registros  valor_par_ponderado  valor_par_promedio_simple  participacion_valor_pct  participacion_pares_pct
Código_SA6 Flujo_comercial                                                                                                                                                                         
640399     Exportación              8,514.24                  320.75                30         40                26.54                      33.11                    77.99                    78.09
           Importación             11,709.68                  497.50                30         40                23.54                      27.13                    63.27                    66.24
640411     Exportación              2,173.93                   84.49                30         40                25.73                      27.51                    19.91                    20.57
           Importación              6,447.48                  245.87                30         40                26.22                      30.05                    34.84                    32.74
640420     Exportación                229.19                    5.53                30         40                41.42                      44.72                     2.10                     1.35
           Importación                348.93                    7.72                30         40                45.22                      55.97                     1.89                     1.03

In [3]:
def ranking_socios(df, flujo, metrica, n=5):
    serie = df[df["Flujo_comercial"] == flujo].groupby("Socio_comercial")[metrica].sum().nlargest(n)
    escala = 1e6
    return (serie / escala).round(1).rename("millones")

for flujo in ["Importación", "Exportación"]:
    print(f"=== {flujo}: top 5 por VALOR (millones de USD) ===")
    print(ranking_socios(df, flujo, "Valor_USD").to_string())
    print(f"=== {flujo}: top 5 por CANTIDAD (millones de pares) ===")
    print(ranking_socios(df, flujo, "Cantidad_pares").to_string())
    print()

=== Importación: top 5 por VALOR (millones de USD) ===
Socio_comercial
Grecia         2,109.60
Países Bajos   1,754.10
Vietnam        1,517.60
Irlanda          912.70
Turquía          908.00
=== Importación: top 5 por CANTIDAD (millones de pares) ===
Socio_comercial
Países Bajos      85.50
Grecia            80.50
República Checa   64.40
Vietnam           52.10
Finlandia         38.10

=== Exportación: top 5 por VALOR (millones de USD) ===
Socio_comercial
Indonesia   2,286.30
Vietnam       786.00
Eslovenia     749.40
Irlanda       606.30
Grecia        553.20
=== Exportación: top 5 por CANTIDAD (millones de pares) ===
Socio_comercial
Indonesia   104.70
Vietnam      37.10
Eslovenia    26.90
Irlanda      21.50
Austria      21.00



In [4]:
comp = df.groupby(["Flujo_comercial", "Socio_comercial"]).agg(valor=("Valor_USD", "sum"), pares=("Cantidad_pares", "sum"))
comp["ranking_valor"] = comp.groupby("Flujo_comercial")["valor"].rank(ascending=False, method="min").astype(int)
comp["ranking_cantidad"] = comp.groupby("Flujo_comercial")["pares"].rank(ascending=False, method="min").astype(int)
comp["cambio_posiciones"] = comp["ranking_cantidad"] - comp["ranking_valor"]
comp["valor_por_par"] = comp["valor"] / comp["pares"]
comp["valor_musd"] = comp["valor"] / 1e6
comp["pares_millones"] = comp["pares"] / 1e6
cols = ["ranking_valor", "ranking_cantidad", "cambio_posiciones", "valor_musd", "pares_millones", "valor_por_par"]

for flujo in ["Importación", "Exportación"]:
    sub = comp.loc[flujo, cols]
    cambios = sub[sub["cambio_posiciones"].abs() >= 3].sort_values("ranking_valor")
    print(f"=== {flujo}: socios que cambian 3 o más posiciones al pasar de valor a cantidad ===")
    display(cambios.round(1))
    coinciden = len(set(sub.nsmallest(5, "ranking_valor").index) & set(sub.nsmallest(5, "ranking_cantidad").index))
    print(f"Socios que están en el top 5 por valor y también en el top 5 por cantidad: {coinciden} de 5\n")

=== Importación: socios que cambian 3 o más posiciones al pasar de valor a cantidad ===


,ranking_valor,ranking_cantidad,cambio_posiciones,valor_musd,pares_millones,valor_por_par
Socio_comercial,,,,,,
Irlanda,4,14,10,912.70,21.70,42.00
Turquía,5,8,3,908.00,26.10,34.70
Camboya,9,15,6,724.80,21.70,33.50
República Checa,10,3,-7,595.80,64.40,9.30
China,13,9,-4,563.50,26.10,21.60
Croacia,15,10,-5,486.90,25.60,19.00
Alemania,16,12,-4,477.90,22.90,20.80
Suecia,17,21,4,458.20,14.70,31.20
Marruecos,22,19,-3,395.20,15.60,25.40


Socios que están en el top 5 por valor y también en el top 5 por cantidad: 3 de 5

=== Exportación: socios que cambian 3 o más posiciones al pasar de valor a cantidad ===


,ranking_valor,ranking_cantidad,cambio_posiciones,valor_musd,pares_millones,valor_por_par
Socio_comercial,,,,,,
Italia,7,11,4,529.00,12.60,41.80
Países Bajos,12,17,5,290.90,8.70,33.40
Hungría,15,12,-3,248.20,12.30,20.10
Bélgica,17,14,-3,226.90,9.60,23.60


Socios que están en el top 5 por valor y también en el top 5 por cantidad: 4 de 5



## Síntesis ejecutiva

**Tres hallazgos**

1. **El calzado importado supera al exportado en el mercado de referencia.** **[Dato]** En 2020-2024, España registra 18.506 millones de USD importados frente a 10.917 millones exportados (1,7×) y 751 millones de pares frente a 411 millones (1,8×). **[Conclusión]** La presión importadora es estructural; para Manaco, la defensa del mercado interno requiere diferenciación y control de costos, no solo precio.
2. **El comercio se concentra en una categoría y el valor unitario separa un nicho.** **[Dato]** La categoría 640399 reúne 63,3% del valor importado y 78,0% del exportado; el deportivo (640411) pesa 34,8% de las importaciones frente a 19,9% de las exportaciones. La categoría 640420 representa menos de 2,2% del valor, pero su valor por par es 45,22 USD (importación) y 41,42 USD (exportación), frente a 23,5-26,5 USD en las otras dos. **[Conclusión]** Las líneas de volumen compiten en un rango estrecho de valor aduanero y las líneas de cuero son un nicho de mayor valor unitario.
3. **Valor y cantidad cuentan historias distintas.** **[Dato]** En importación, Irlanda ocupa el puesto 4 por valor (913 millones de USD) y el 14 por pares (21,7 millones): 42,0 USD/par; República Checa es la tercera por pares (64,4 millones) pero la décima por valor (596 millones): 9,3 USD/par. En exportación el top 5 por valor y por cantidad coincide en 4 de 5 socios. **[Conclusión]** Un ranking por una sola métrica oculta si el competidor vende volumen barato o producto de mayor valor; Manaco debe monitorear ambas.

**Dos limitaciones**

1. **No hay datos internos de Manaco** (ventas, costos, inventario, defectos, devoluciones, clientes) ni datos de Bolivia: la base es de España y es una muestra de 240 registros. Toda extrapolación a Manaco es un **[Supuesto]** por analogía.
2. **El valor por par no es el precio al consumidor:** es un valor aduanero (FOB/CIF) que excluye márgenes, impuestos, retail y logística; no permite estimar rentabilidad ni precios de venta.

**Una pregunta estratégica para Manaco**

¿Qué costo por par y qué margen unitario necesita cada línea de Manaco para competir en el mercado boliviano frente a importaciones cuyo valor aduanero de referencia ronda 24-27 USD por par en las categorías de mayor volumen y supera los 40 USD en el nicho de cuero?